# Schema validation and join primitives

This notebook covers schema validation utilities and the two join primitives.

| # | Topic |
|---|---|
| 1 | `infer_schema_map` — infer dtype map from a DataFrame |
| 2 | `normalize_dtype_alias` / `validate_schema` — dtype comparison and validation |
| 3 | `apply_schema_map` — cast DataFrame columns to canonical dtypes |
| 4 | `align_frames_for_join` — normalize two frames for merging |
| 5 | `left_join_frames` / `indexed_left_join` — join primitives with dtype alignment |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd

from boti_data.schema import (
    SchemaValidationError,
    align_frames_for_join,
    apply_schema_map,
    infer_schema_map,
    normalize_dtype_alias,
    normalize_schema_map,
    validate_schema,
)
from boti_data import left_join_frames, indexed_left_join

## 1. `infer_schema_map` — infer dtype map from a DataFrame

Produces a normalized `{column: dtype}` dictionary from a DataFrame.

In [2]:
df = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "score": [95.5, 87.0, 92.3],
    "active": [True, False, True],
    "created": pd.to_datetime(["2024-01-01", "2024-02-01", "2024-03-01"]),
})

schema = infer_schema_map(df)
for col, dtype in schema.items():
    print(f"  {col:12s} -> {dtype}")

# Infer only specific columns
partial = infer_schema_map(df, columns=["name", "score"])
print(f"\nPartial schema: {partial}")

  id           -> Int64
  name         -> str
  score        -> Float64
  active       -> boolean
  created      -> datetime64[ns]

Partial schema: {'name': 'str', 'score': 'Float64'}


## 2. `normalize_dtype_alias` / `validate_schema` — dtype comparison and validation

`normalize_dtype_alias` converts dtype strings to canonical form so that semantically identical types compare equal. `validate_schema` checks a DataFrame matches an expected schema.

In [3]:
# Normalize various alias forms
aliases = ["int64", "Int64", "int64[pyarrow]", "float64", "double[pyarrow]",
          "string[python]", "string[pyarrow]", "boolean", "bool",
          "datetime64[ns, utc]", "datetime64[us, utc]", "timestamp[ns, tz=utc][pyarrow]"]
for alias in aliases:
    print(f"  {alias:45s} -> {normalize_dtype_alias(alias)}")

print()

# Validate a DataFrame against a schema
expected = {"id": "Int64", "name": "string", "score": "Float64"}
try:
    validate_schema(df, expected)
    print("Schema validation: PASSED")
except SchemaValidationError as e:
    print(f"Schema validation: FAILED\n{e}")

  int64                                         -> Int64
  Int64                                         -> Int64
  int64[pyarrow]                                -> Int64
  float64                                       -> Float64
  double[pyarrow]                               -> Float64
  string[python]                                -> string
  string[pyarrow]                               -> string
  boolean                                       -> boolean
  bool                                          -> boolean
  datetime64[ns, utc]                           -> datetime64[ns, UTC]
  datetime64[us, utc]                           -> datetime64[ns, UTC]
  timestamp[ns, tz=utc][pyarrow]                -> datetime64[ns, UTC]

Schema validation: FAILED
Schema validation failed:
Column 'name': expected 'string', found 'str'.


In [4]:
# Validation failure example
df_wrong = pd.DataFrame({"id": ["1", "2"], "name": ["x", "y"], "score": ["bad", "data"]})
try:
    validate_schema(df_wrong, expected)
except SchemaValidationError as e:
    print(f"Validation error:\n{e}")

Validation error:
Schema validation failed:
Column 'id': expected 'Int64', found 'str'.
Column 'name': expected 'string', found 'str'.
Column 'score': expected 'Float64', found 'str'.


## 3. `apply_schema_map` — cast DataFrame columns to canonical dtypes

Coerces columns to the target dtypes, handling edge cases like boolean literals, timezone-aware datetimes, and numeric coercion.

In [5]:
raw_df = pd.DataFrame({
    "id": ["1", "2", "3"],
    "active": ["true", "false", "yes"],
    "created": ["2024-01-01", "invalid", "2024-03-01"],
})

target_schema = {
    "id": "Int64",
    "active": "boolean",
    "created": "datetime64[ns, UTC]",
}

cast_df = apply_schema_map(raw_df, target_schema)
print(cast_df.dtypes)
print()
print(cast_df)

id                       Int64
active                 boolean
created    datetime64[us, UTC]
dtype: object

   id  active                   created
0   1    True 2024-01-01 00:00:00+00:00
1   2   False                       NaT
2   3    True 2024-03-01 00:00:00+00:00


## 4. `align_frames_for_join` — normalize two frames for merging

Ensures both frames have identical, validated dtypes on the join key columns.

In [6]:
left = pd.DataFrame({"key": ["1", "2", "3"], "value_l": [10, 20, 30]})
right = pd.DataFrame({"key": [1, 2, 4], "value_r": ["a", "b", "c"]})

# The join key must have the same dtype on both sides
join_schema = {"key": "Int64"}

left_aligned, right_aligned = align_frames_for_join(left, right, join_schema)
print(f"Left key dtype:  {left_aligned.dtypes['key']}")
print(f"Right key dtype: {right_aligned.dtypes['key']}")
print(f"\nLeft aligned:\n{left_aligned}\n")
print(f"Right aligned:\n{right_aligned}")

Left key dtype:  Int64
Right key dtype: Int64

Left aligned:
   key
0    1
1    2
2    3

Right aligned:
   key
0    1
1    2
2    4


## 5. `left_join_frames` / `indexed_left_join` — join primitives

These high-level join wrappers first align dtypes, then perform the join.

In [7]:
# left_join_frames with schema alignment
left = pd.DataFrame({"id": ["1", "2", "3"], "name": ["a", "b", "c"]})
right = pd.DataFrame({"id": [1, 2], "score": [95, 87]})

joined = left_join_frames(
    left, right,
    left_on=["id"],
    right_on=["id"],
    join_schema_map={"id": "Int64"},
)
print("left_join_frames result:")
print(joined)
print()

# Multi-key join (same key names on both sides)
left2 = pd.DataFrame({
    "org_id": [1, 1, 2],
    "user_id": ["10", "20", "30"],
    "name": ["x", "y", "z"],
})
right2 = pd.DataFrame({
    "org_id": [1, 2],
    "user_id": ["10", "30"],
    "role": ["admin", "viewer"],
})

joined2 = left_join_frames(
    left2, right2,
    left_on=["org_id", "user_id"],
    right_on=["org_id", "user_id"],
    join_schema_map={"org_id": "Int64", "user_id": "Int64"},
)
print("Multi-key join result:")
print(joined2)

left_join_frames result:
   id name  score
0   1    a   95.0
1   2    b   87.0
2   3    c    NaN

Multi-key join result:
   org_id  user_id name    role
0       1       10    x   admin
1       1       20    y     NaN
2       2       30    z  viewer


In [8]:
# indexed_left_join — preferred for large Dask joins
left3 = pd.DataFrame({
    "key": [1, 2, 3, 4],
    "value": [10, 20, 30, 40],
})
right3 = pd.DataFrame({
    "key": [1, 2, 4],
    "label": ["a", "b", "d"],
})

joined3 = indexed_left_join(
    left3, right3,
    join_key="key",
    join_schema_map={"key": "Int64"},
    reset_index=True,
)
print("indexed_left_join result:")
print(joined3)
print(f"\nIndex reset: {'key' in joined3.columns}")

indexed_left_join result:
   key  value label
0    1     10     a
1    2     20     b
2    3     30   NaN
3    4     40     d

Index reset: True


### Summary

- **`infer_schema_map`** / **`validate_schema`** — lightweight DataFrame schema contracts.
- **`normalize_dtype_alias`** — canonicalizes dtype strings across pandas, PyArrow, and Dask.
- **`apply_schema_map`** — coerces columns to target dtypes with boolean literal parsing, UTC datetime handling, and numeric coercion.
- **`left_join_frames`** / **`indexed_left_join`** — schema-safe join wrappers that align dtypes before merging (prefer `indexed_left_join` for large Dask joins).